# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

<div style="background-color:#00000b">

_Fill in your group number **from Brightspace**, names, and student numbers._
    
|    Group   |           X          |
|------------|----------------------|
| Student A  |        XXXXXXX       |
| Student B  |        XXXXXXX       |
| Student C  |        XXXXXXX       |
| Student D  |        XXXXXXX       |

#### Imports

In [4]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import numpy as np
import random
import sys
import time

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData

## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

There are multiple variants of the TSP problem. The classic problem may be formally formulated as follows:
<br>
>Given a complete graph $G=(V, A)$, and a distance function $d: A \rightarrow$ $R^+$, we are required to find a Hamiltonian circuit of minimal cost. A Hamiltonian circuit is a cycle that visits every node exactly once, except the first node. 
<br>

__[Extra]__: Given a fixed endpoint, this can be converted to a problem of finding the Hamiltonian path by adding a dummy node, connected only to the endpoint (all other nodes have very high cost on these connections, such that they may be ignore). If we are indifferent to any specific endpoint, we either remove the dummy node entirely, or connect it to all nodes with a cost of $0$.
<br>

__[Errata]__: Our problem differs in a few ways from the definition given above. See below. (The graph may be constructed to avoid these issues, however.)

#### Question 2

__Differences:__
1. We do not need to find a minimum cost circuit, a path is enough - that is, the robot needn't return to its starting position;
1. The robot may pass through the same node (product) multiple times on its optimal path;
1. The start point and end point are fixed. We only care about what is in the middle (i.e. the products), so that the chromosome will stay the same, but the fitness calculator will also have to add the cost from the start to the first product and from the last product to the end.

#### Question 3

[NOT ANSWERED]

### 1.2 Genetic Algorithm

In [13]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:
    rng = None
    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    @param x_prob:
    @param mut_prob:
    @param random_seed:
    """
    def __init__(self, generations, pop_size, x_prob, mut_prob, random_seed, elitist):
        self.generations = generations
        self.pop_size = pop_size
        self.x_prob = x_prob
        self.mut_prob = mut_prob
        self.rng = np.random.default_rng(random_seed)
        self.elitist = elitist

    """
    This method gives a score to a certain chromosome.
    @param tsp_data: the data describing the problem.
    @param chromosome: a permutation of the products (a certain order of visiting them) (0-indexed)
    """
    def cost(self, tsp_data, chromosome):
        size = len(chromosome)
        #chromosome = np.asarray(chromosome).astype(np.int64)
        assert not isinstance(chromosome[0], float), "[FTINESS] Chromosome composed of floats instead of ints"
        cost = tsp_data.start_distances[chromosome[0]]
        for i in range(size - 1):
            cost += tsp_data.distances[chromosome[i]][chromosome[i + 1]]

        cost += tsp_data.end_distances[chromosome[size - 1]] + size
        assert cost >= 0, "[FITNESS] Cost becomes negative"
        return cost

    """
    Generates the members of the original population - random permutations. Seed of random given above for reproducibility.
    @param tsp_data: the data describing the problem.
    """
    def initialize_population(self, tsp_data):
        # amount of products => size of chromosome
        product_count = len(tsp_data.product_locations)
        pops = [self.rng.permutation(product_count) for i in range(self.pop_size)]
        return pops

    """
    This method implements the Order crossover operator (OX1 for short).
    @param p_1: parent 1.
    @param p_2: parent 2.
    @return the generated offspring (I'll only generate one offspring).
    """
    def ox1(self, p_1, p_2):
        n = len(p_1)
        # pick random tour from parent 1
        start = self.rng.integers(0, n)
        end = self.rng.integers(start + 1, n + 1) # exclusive
        visited = [False for i in range(n)]
        offspring = [None for i in range(n)]
        # insert (sub)tour
        for i in range(start, end):
            offspring[i] = p_1[i]
            visited[p_1[i]] = True

        # rest (starting with part after tour)
        idx_p2 = end - start
        for i in range(n - (end - start)):
            while(visited[p_2[idx_p2]]):
                idx_p2 = (idx_p2 + 1) % n

            offspring[(end + i) % n] = p_2[idx_p2]
            visited[p_2[idx_p2]] = True

        return offspring

    """
    This method implements the Genetic Edge Recombination crossover operator (ER for short). Don't use because it sucks.
    @param p_1: parent 1.
    @param p_2: parent 2.
    @return the generated offspring.
    """
    def er(self, p_1, p_2):
        n = len(p_1)
        # create edge map for the products
        # I assume first product is not circularly connected with the last
        edge_map = [[] for i in range(n)]
        for i in range(n):
            pid1 = p_1[i]
            pid2 = p_2[i]
            # +1 because of count_nonzero
            if i > 0:
                edge_map[pid1].append(p_1[i - 1] + 1)
                edge_map[pid2].append(p_2[i - 1] + 1)

            if i < n - 1:
                edge_map[pid1].append(p_1[i + 1] + 1)
                edge_map[pid2].append(p_2[i + 1] + 1)

        # create offspring from parents
        child = [None] * n
        included = [False] * n
        crt = p_1[0] # deterministically pick first from p_1, doesn't really matter
        included[crt] = True
        child[0] = crt
        for i in range(1, n):
            # smallest amount of edges left (first found breaks ties)
            low = np.argmin([((n + 1) if included[j] else np.count_nonzero(edge_map[j])) for j in range(n)])
            included[low] = True
            edge_map = [list(filter(lambda x: x != low, edge_map[j])) for j in range(n)] # remove all occurrences of this product
            child[i] = low

        return child
    
    """
    Mutates the chromosome with the Inversion Mutation (IVM) operator.
    """
    def ivm(self, chromosome):
        n = len(chromosome)
        # first pick a random tour
        start = self.rng.integers(0, n)
        end = self.rng.integers(start + 1, n + 1) # end is exclusive

        # (subtour is also reversed)
        tour = chromosome[start:end]

        # remove tour from chromosome
        mut_c = np.r_[chromosome[0:start], chromosome[end:n]]
        assert len(mut_c) == n - (end - start), "[IVM] Partial step produces mutated chromosome of unexpected length"
        # pick random position to insert the tour
        idx = self.rng.integers(0, len(mut_c) + 1)
        mut_c = np.insert(mut_c, idx, tour[::-1])
        assert len(mut_c) == n, "[IVM] Length of mutated chromosome doesn't match original length"
        return mut_c
        
    """
    Generate offspring from the given parents, using the ER operator
    @param p_1: parent 1.
    @param p_2: parent 2.
    @return the generated offspring.
    """
    def generate_offspring(self, tsp_data, parents, costs):
        n = self.pop_size
        mp = self.mut_prob
        xp = self.x_prob

        cost_idx = [(costs[i], i) for i in range(n)]
        cost_idx = sorted(cost_idx, key=lambda x: x[0])
        t_size = 5 # amount of participants in tournament
        new_gen = [None for i in range(n)]
        for i in range(n):
            # pick two random parents by means of tournament
            c_val_1 = n
            c_val_2 = n
            for part in range(t_size):
                c_val_1 = min(c_val_1, self.rng.integers(0, n))
                c_val_2 = min(c_val_2, self.rng.integers(0, n))

            parent_1 = parents[cost_idx[c_val_1][1]]
            parent_2 = parents[cost_idx[c_val_2][1]]
            new_gen[i] = parent_1 # default in case no crossover happens

            # crossover
            if self.rng.random() < xp:
                new_gen[i] = self.ox1(p_1=parent_1, p_2=parent_2)

            # mutation
            if self.rng.random() < mp:
                new_gen[i] = self.ivm(new_gen[i])

            if isinstance(new_gen[i], np.ndarray):
                new_gen[i] = new_gen[i].tolist()

            # casting bonanza
            for j in range(len(new_gen[i])):
                new_gen[i][j] = int(new_gen[i][j])

            assert not isinstance(new_gen[i][0], float), "[OFFSPRING GENERATION] tf"
        
        return new_gen

    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        n = self.pop_size

        # store the best at every step
        best_cost = float("inf")
        best = None

        # store original mutation prob
        orig_mut_p = self.mut_prob
        lowest_costs = []
        avg_costs = []

        # amount of (top) performers from current generation that are kept
        peaks = 5

        population = self.initialize_population(tsp_data)
        assert len(population) == n, "[SOLVE TSP] Population generated of size different than what is expected"
        for _ in range(self.generations):
            assert not isinstance(population[0][0], float), "[SOLVE TSP] Pop contains floats instead of ints"
            costs = [self.cost(tsp_data, population[i]) for i in range(n)]
            # best member of current generation is compared to best of all time
            c_cost = np.min(costs)
            lowest_costs.append(c_cost)
            avg_c_cost = np.mean(costs)
            avg_costs.append(avg_c_cost)
            #print(c_cost, avg_c_cost)
            if c_cost < best_cost:
                best_cost = c_cost
                best = population[np.argmin(costs)]

            # store top #peak (different) performers
            # (by means of roulette wheel selection)
            scores = [(10000 / costs[j]) for j in range(n)]
            total_fit = np.sum(scores)
            cummulative = [0 for i in range(n)]
            # calculate cummulative scores
            for i in range(n):
                if i == 0:
                    cummulative[i] = scores[i]
                else:
                    cummulative[i] = cummulative[i - 1] + scores[i]

            top = [None for i in range(peaks)]
            # top performer is picked with certainty
            if peaks > 0:
                top[0] = population[np.argmin(costs)]
                
            for i in range(1, peaks):
                cut = total_fit * self.rng.random()
                index = np.min(np.where(cummulative > cut))
                top[i] = population[index]
            
            # generate offspring
            population = self.generate_offspring(tsp_data, population, costs)
            if self.elitist:
                for i in range(peaks):
                    population[i] = top[i]

            # adaptive elitism and mutation rate
            if _ > 100:
                # increase mutation probability if we get stuck in a local optimum
                if lowest_costs[_ - 101] == lowest_costs[_ - 1]:
                    self.mut_prob = max(self.mut_prob * 1.03, 0.5)
                    peaks = max(peaks - 1, 0)
                else: # until the lowest cost decreases again
                    self.mut_prob = orig_mut_p
                    peaks = min(peaks + 1, 5)

        self.mut_prob = orig_mut_p
        return best 

In [14]:
random_seed = 1337

In [18]:
data = TSPData.read_from_file("../data/optimal_tsp")
ga = GeneticAlgorithm(1000, 100, 0.9, 0.05, random_seed, True)
best = ga.solve_tsp(data)
print(best, ga.cost(data, best))

[0, 1, 6, 4, 13, 15, 3, 8, 17, 7, 9, 14, 11, 12, 5, 10, 2, 16] 1343


#### Question 4

In my representation, a chromosome will represent a Hamiltonian tour in the following way: it will be composed of genes representing specific products (in the classic problem, cities), where the order of the genes gives the order in which the products are taken.
<br>
As an example, if we had 5 cities (numbered from 1 to 5), a certain tour may be encoded as a chromosome in the following way:
<br>
>$[1, 5, 3, 4, 2]$,

where the numbers represent genes.

#### Question 5

The fitness function will simply be the inverse of the cost of the given tour (1 over the sum of distances between all consecutive products). This is exactly the objective we try to maximize in TSP and represents, thus, a good way to measure fitness (we want to minimze the cost and, hence, maximize 1/cost).

#### Question 6

Parents are selected by means of tournament (which seemed to perform better than roulette wheel selection - but which I still nevertheless use in the elitism selection). A tournament of size 5 worked best from the values I tested.

#### Question 7

I implemented 2 different crossover operators and one mutation operator.
<br>Crossover:
- Genetic Edge Recombination crossover (ER), which gave the best results in the paper I used for research, but which ultimately seemed to perform poorly here (likely because the implementation is harder to make efficient);
- Order crossover (OX1), which performed significantly better (and faster) than the former.

Mutation:
- Inversion Mutation (IVM) operator, which, according to the paper, performs the best (though doesn't necessarily converge the quickest). Still, in practice, it works fine.

#### Question 8

The manner through which I avoid local minima is (at least) two-fold: I use mutation (with a relatively low probability of occurence initially) and adaptive probability (if 100 generations ago the lowest cost was the same as current one, I increase the mutation probability multiplicatively by 1.03).

#### Question 9

Elitism, in the context of genetic algorithms, is a technique revolving around the persistence of certain members of the current population into the following generation(s), directly.

In my implementation, elitism is optional (a True/False parameter). I believe it's not necessarily needed, but because the objective (or the topography of the search space for possible solutions) doesn't change it's a good way of reaching convergence quicker.

#### Question 10

In [17]:
# Please keep your parameters for the Genetic Algorithm easily changeable here
population_size = 80
generations = 600
persist_file = "./../data/optimal_tsp"

# Setup optimization
tsp_data = TSPData.read_from_file(persist_file)
ga = GeneticAlgorithm(generations, population_size, 0.9, 0.05, random_seed, True)

# Run optimzation and write to file
solution = ga.solve_tsp(tsp_data)
tsp_data.write_action_file(solution, "./../data/tsp_solution.txt")
print(solution, ga.cost(tsp_data, solution))

[0, 1, 6, 4, 13, 15, 3, 8, 7, 17, 9, 14, 11, 12, 5, 10, 2, 16] 1343


The length of my solution is $1343$. In all my iterations, regardless of (adaptable) mutation rate, elitism, crossover probability, tournament size, random seed, population size and number of generations, I have never gotten a better value. If this is not optimal, it (should be) is at the very least quite close to this value (and the true optimal must then be somehow quite different from the solution I've gotten).

[Needs visualization(s)]

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 12

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 13

<div style="background-color:#f1be3e">

_Write your answer here. You can also use LaTeX notation in a Jupyter Notebook._

#### Question 14

<div style="background-color:#f1be3e">

_Write your answer here. You can also use LaTeX notation in a Jupyter Notebook._

### 2.3 Implementing the Ant Algorithm

In [9]:
# Class that represents the basic Ant functionality
class StandardAnt:

    """
    Constructor of a StandardAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


In [10]:
# Class that holds all of the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ints representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length):
        self.walls = walls
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self):
        pass

    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        pass

    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        pass

    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        pass

    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the poition of interest
    @return the amount of pheromone at the specified poition
    """
    def get_pheromone(self, pos):
        pass

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            return Maze(maze_layout, width, length)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [11]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification):
        self.maze.reset()
        pass

In [12]:
# Please keep your parameters for the ACO easily changeable here
gen = 1
no_gen = 1
q = 1600
evap = 0.1

# Construct the optimization objects
maze = Maze.create_maze("./../data/hard_maze.txt")
spec = PathSpecification.read_coordinates("./../data/hard_coordinates.txt")
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
print("Route size: " + str(shortest_route.size()))

shortest_route.write_to_file("./../data/hard_solution.txt")

Ready reading maze file ./../data/hard_maze.txt
Time taken: 0.0


AttributeError: 'NoneType' object has no attribute 'size'

### 2.4 Upgrading Your Ants with Intelligence

#### Question 15

In [0]:
# Class that represents the intelligent Ant
class IntelligentAnt:

    """
    Constructor of an IntelligentAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [0]:
# Please keep your parameters for the synthesis part easily changeable here
gen = 1
no_gen = 1
q = 1000
evap = 0.1

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt")
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#f1be3e">

**If you made use of any non-course resources, cite them below.**